In [1]:
import gc

In [2]:
import numpy as np
import pandas as pd
import xarray as xr

In [3]:
from data import ds_single, ds_pressure, _load_single_dataset

In [4]:
from dataclasses import dataclass
from enum import StrEnum


class ErrorKind(StrEnum):
    POINTWISE = "pointwise"
    MEAN = "mean"

    def __repr__(self) -> str:
        return f"{type(self).__name__}.{self._name_}"


@dataclass
class ExpertEnsembleSpreadQuantile:
    absolute: float
    absolute_value: float
    absolute_kind: ErrorKind
    relative: float
    relative_value: float
    relative_kind: ErrorKind

In [5]:
def compute_ensemble_spread_quantile(
    mean: xr.DataArray,
    spread: xr.DataArray,
    expert_abs: float,
    expert_abs_kind: ErrorKind,
    expert_rel: float,
    expert_rel_kind: ErrorKind,
) -> ExpertEnsembleSpreadQuantile:
    if np.isnan(expert_abs) and np.isnan(expert_rel):
        return ExpertEnsembleSpreadQuantile(
            absolute=np.nan,
            absolute_value=expert_abs,
            absolute_kind=expert_abs_kind,
            relative=np.nan,
            relative_value=expert_rel,
            relative_kind=expert_rel_kind,
        )

    mean = mean.values.flatten()
    spread = spread.values.flatten()

    spread_nonzero = spread[spread > 0.0]

    if (len(spread_nonzero) > 0) and (not np.isnan(expert_abs)):
        absolute = 1 - np.searchsorted(np.sort(spread_nonzero), expert_abs) / len(
            spread_nonzero
        )
    else:
        absolute = np.nan

    abs_mean = np.abs(mean)
    rel = spread[abs_mean > 0.0] / abs_mean[abs_mean > 0.0]
    rel_nonzero = rel[rel > 0.0]

    if (len(rel_nonzero) > 0) and (not np.isnan(expert_rel)):
        relative = 1 - np.searchsorted(np.sort(rel_nonzero), expert_rel) / len(
            rel_nonzero
        )
    else:
        relative = np.nan

    return ExpertEnsembleSpreadQuantile(
        absolute=absolute,
        absolute_value=expert_abs,
        absolute_kind=expert_abs_kind,
        relative=relative,
        relative_value=expert_rel,
        relative_kind=expert_rel_kind,
    )

In [6]:
from tqdm import tqdm

In [7]:
expert = pd.read_csv("expert.csv")

In [8]:
def parse_float(x: str):
    if len(x.strip()) == 0:
        return np.nan
    return float(x.replace(",", "."))

In [9]:
with open("expert-quantile-single.txt", "w") as f:
    i = 0
    for key in tqdm(sorted(ds_single.mean)):
        if (i > 0) and (i % 50 == 0):
            del ds_single
            gc.collect()
            ds_single = _load_single_dataset()
        ex = expert[
            (expert["Short Name"] == key) & (expert["Level"] == "Single/Surface")
        ]
        if len(ex) > 0:
            assert len(ex) == 1
            ex = ex.iloc[0]
            absl1 = parse_float(str(ex["Absolute L1 error"]))
            abslinf = parse_float(str(ex["Absolute Linfty error"]))
            if np.isnan(absl1) and np.isnan(abslinf):
                exabsmax = np.nan
                exabskind = ErrorKind.POINTWISE
            elif np.isnan(absl1):
                exabsmax = abslinf
                exabskind = ErrorKind.POINTWISE
            elif np.isnan(abslinf):
                exabsmax = absl1
                exabskind = ErrorKind.MEAN
            elif absl1 >= abslinf:
                exabsmax = abslinf
                exabskind = ErrorKind.POINTWISE
            else:
                print(f"expert abs mismatch {key}: L1={absl1} Linfty={abslinf}")
                exabsmax = absl1
                exabskind = ErrorKind.MEAN
            rell1 = parse_float(str(ex["Relative L1 error"]))
            rellinf = parse_float(str(ex["Relative Linfty error"]))
            if np.isnan(rell1) and np.isnan(rellinf):
                exrelmax = np.nan
                exrelkind = ErrorKind.POINTWISE
            elif np.isnan(rell1):
                exrelmax = rellinf
                exrelkind = ErrorKind.POINTWISE
            elif np.isnan(rellinf):
                exrelmax = rell1
                exrelkind = ErrorKind.MEAN
            elif rell1 >= rellinf:
                exrelmax = rellinf
                exrelkind = ErrorKind.POINTWISE
            else:
                print(f"expert rel mismatch {key}: L1={rell1} Linfty={rellinf}")
                exrelmax = rell1
                exrelkind = ErrorKind.MEAN
        else:
            exabsmax = np.nan
            exabskind = ErrorKind.POINTWISE
            exrelmax = np.nan
            exrelkind = ErrorKind.POINTWISE
        print(f"{key}: ", end="", flush=True, file=f)
        print(
            f"{compute_ensemble_spread_quantile(ds_single.mean[key], ds_single.spread[key], exabsmax, exabskind, exrelmax, exrelkind)!r}",
            flush=True,
            file=f,
        )
        if not (np.isnan(exabsmax) and np.isnan(exrelmax)):
            i += 1

100%|████████████████████████████████████████████████████████| 226/226 [21:24<00:00,  5.68s/it]


In [10]:
with open("expert-quantile-pressure.txt", "w") as f:
    for key in tqdm(sorted(ds_pressure.mean)):
        ex = expert[
            (expert["Short Name"] == key) & (expert["Level"] == "Pressure+Model")
        ]
        if len(ex) > 0:
            assert len(ex) == 1
            ex = ex.iloc[0]
            absl1 = parse_float(str(ex["Absolute L1 error"]))
            abslinf = parse_float(str(ex["Absolute Linfty error"]))
            if np.isnan(absl1) and np.isnan(abslinf):
                exabsmax = np.nan
                exabskind = ErrorKind.POINTWISE
            elif np.isnan(absl1):
                exabsmax = abslinf
                exabskind = ErrorKind.POINTWISE
            elif np.isnan(abslinf):
                exabsmax = absl1
                exabskind = ErrorKind.MEAN
            elif absl1 >= abslinf:
                exabsmax = abslinf
                exabskind = ErrorKind.POINTWISE
            else:
                print(f"expert abs mismatch {key}: L1={absl1} Linfty={abslinf}")
                exabsmax = absl1
                exabskind = ErrorKind.MEAN
            rell1 = parse_float(str(ex["Relative L1 error"]))
            rellinf = parse_float(str(ex["Relative Linfty error"]))
            if np.isnan(rell1) and np.isnan(rellinf):
                exrelmax = np.nan
                exrelkind = ErrorKind.POINTWISE
            elif np.isnan(rell1):
                exrelmax = rellinf
                exrelkind = ErrorKind.POINTWISE
            elif np.isnan(rellinf):
                exrelmax = rell1
                exrelkind = ErrorKind.MEAN
            elif rell1 >= rellinf:
                exrelmax = rellinf
                exrelkind = ErrorKind.POINTWISE
            else:
                print(f"expert rel mismatch {key}: L1={rell1} Linfty={rellinf}")
                exrelmax = rell1
                exrelkind = ErrorKind.MEAN
        else:
            exabsmax = np.nan
            exabskind = ErrorKind.POINTWISE
            exrelmax = np.nan
            exrelkind = ErrorKind.POINTWISE
        print(f"{key}:", flush=True, file=f)
        for pressure in sorted(ds_pressure.mean[key].pressure_level.values):
            print(f" - {pressure} hPa: ", end="", flush=True, file=f)
            print(
                f"{compute_ensemble_spread_quantile(ds_pressure.mean[key].sel(pressure_level=pressure), ds_pressure.spread[key].sel(pressure_level=pressure), exabsmax, exabskind, exrelmax, exrelkind)!r}",
                flush=True,
                file=f,
            )

 94%|██████████████████████████████████████████████████████▍   | 15/16 [07:51<00:46, 46.28s/it]

expert abs mismatch z: L1=5.0 Linfty=10.0


100%|██████████████████████████████████████████████████████████| 16/16 [08:24<00:00, 31.52s/it]
